In [7]:
import os 
from dotenv import load_dotenv
import xml.etree.ElementTree as ET
import io
import requests
import zipfile

In [8]:
load_dotenv()

True

In [9]:
API_KEY = os.getenv('dart')

In [10]:
# 기업 코드 로드 함수 생성 
def get_corp_code_dict():
    url = "https://opendart.fss.or.kr/api/corpCode.xml"
    params = {
        'crtfc_key' : API_KEY
    }
    res = requests.get(url, params=params)
    with zipfile.ZipFile(io.BytesIO(res.content)) as zf:
        xml_content = zf.read(zf.namelist()[0])
    tree = ET.ElementTree(ET.fromstring(xml_content))
    root = tree.getroot()

    corp_dict = {}

    for child in root:
        corp_name = child.findtext('corp_name')
        corp_code = child.findtext('corp_code')
        corp_dict[corp_name] = corp_code
    return corp_dict

In [11]:
corp_dict = get_corp_code_dict()

In [12]:
samsung_corp_code = corp_dict.get("삼성전자")

In [14]:
# 삼성전자 최근 공시 목록 가져오기
url = f"https://opendart.fss.or.kr/api/list.json?crtfc_key={API_KEY}&corp_code={samsung_corp_code}&bgn_de=20240101"
resp = requests.get(url)
data = resp.json()

for report in data['list']:
    print(report['report_nm'], report['rcept_no'])

주식등의대량보유상황보고서(일반) 20260417000682
임원ㆍ주요주주특정증권등소유상황보고서 20260417000440
최대주주등소유주식변동신고서               20260413800802
주식등의대량보유상황보고서(일반) 20260413002934
임원ㆍ주요주주특정증권등소유상황보고서 20260408000038
기업설명회(IR)개최(안내공시)               20260407800120
연결재무제표기준영업(잠정)실적(공정공시)               20260407800002
임원ㆍ주요주주특정증권등소유상황보고서 20260406003839
최대주주등소유주식변동신고서               20260403801246
임원ㆍ주요주주특정증권등소유상황보고서 20260403003368


In [13]:
def get_financial_statement(corp_code, bsns_year='2025', reprt_code='11011'):
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    params = {
        "crtfc_key": API_KEY,
        "corp_code": corp_code,
        "bsns_year": bsns_year,       # 사업연도
        "reprt_code": reprt_code,     # 보고서 구분 (11011: 사업보고서)
        "fs_div": "CFS"               # 연결재무제표(CFS) / 개별재무제표(OLS)
    }

    res = requests.get(url, params=params)
    data = res.json()

    if data['status'] == '000':
        return data['list']
    else:
        print("에러:", data['message'])
        return []

financials = get_financial_statement(samsung_corp_code)
for item in financials:  # 예시로 5개만 출력
    print(f"{item['sj_nm']} | {item['account_nm']} = {item['thstrm_amount']}")

재무상태표 | 자산총계 = 566942110000000
재무상태표 | 유동자산 = 247684612000000
재무상태표 | 미수금 = 7481327000000
재무상태표 | 현금및현금성자산 = 57856378000000
재무상태표 | 단기상각후원가금융자산 = 0
재무상태표 | 단기당기손익-공정가치금융자산 = 25715000000
재무상태표 | 선급비용 = 3627172000000
재무상태표 | 매출채권 = 51127642000000
재무상태표 | 재고자산 = 52636828000000
재무상태표 | 매각예정분류자산 = 0
재무상태표 | 기타유동자산 = 6964529000000
재무상태표 | 단기금융상품 = 67965021000000
재무상태표 | 비유동자산 = 319257498000000
재무상태표 | 이연법인세자산 = 18840559000000
재무상태표 | 무형자산 = 29480565000000
재무상태표 | 관계기업 및 공동기업 투자 = 13772121000000
재무상태표 | 당기손익-공정가치금융자산 = 1280501000000
재무상태표 | 기타포괄손익-공정가치금융자산 = 16295005000000
재무상태표 | 순확정급여자산 = 4271547000000
재무상태표 | 기타비유동자산 = 20012416000000
재무상태표 | 유형자산 = 215304784000000
재무상태표 | 자본총계 = 436320337000000
재무상태표 | 지배기업 소유주지분 = 424313255000000
재무상태표 | 기타자본항목 = 16876248000000
재무상태표 | 자본금 = 897514000000
재무상태표 | 보통주자본금 = 778047000000
재무상태표 | 우선주자본금 = 119467000000
재무상태표 | 이익잉여금 = 402135600000000
재무상태표 | 주식발행초과금 = 4403893000000
재무상태표 | 비지배지분 = 12007082000000
재무상태표 | 부채와자본총계 = 566942110000000
재무상태표 | 유동부채 = 